# Sommelier — nghe thử bước tách nhạc (PANNs SED + BS-RoFormer)Notebook này **chỉ chạy đến hết giai đoạn nhạc** rồi dừng, và đóng gói audio để tải về nghe trước/sau.Luồng thực sự chạy:1. **PANNs SED** gán nhãn từng khung 10 ms → `music` / `singing` / `song`2. **BS-RoFormer** bóc nhạc nền ra khỏi các đoạn `music` (giữ lại tiếng nói)3. **Cắt** các đoạn `singing` và `song` ra khỏi bản ghi, nối mép bằng overlap-add equal-power 30 msDừng lại nhờ `steps.diarization = false` trong `config.json` — không có `--stop_after`.**Kết quả tải về** (`music_stage_audio.zip`):| | ||---|---|| `before/` | file gốc || `after/after_music.flac` | bản ghi sau khi bóc nhạc + cắt || `clips/` | từng sự kiện: đoạn trước và đoạn sau, để so trực tiếp || `music_map.json` | mọi span, kèm timeline của phần bị cắt |**GPU:** 1×T4 là đủ. **Thời gian:** cài ~8–12 phút, chạy ~1–3 phút cho mỗi 10 phút audio.

## 1. Kiểm tra môi trường

In [ ]:
import osos.environ["MPLBACKEND"] = "Agg"!nvidia-smi --query-gpu=name,memory.total --format=csv!df -h /kaggle/working 2>/dev/null | tail -1import sys; print("Python:", sys.version.split()[0])

## 2. Thư mục làm việc và repo

In [ ]:
import os, shutilBASE_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else os.getcwd()PROJECT_DIR  = os.path.join(BASE_DIR, 'sommerlier')PIPELINE_DIR = os.path.join(PROJECT_DIR, 'podcast-pipeline')ENV_DIR      = os.path.join(BASE_DIR, 'sommelier_env')AUDIO_DIR    = os.path.join(BASE_DIR, 'vi_audio')OUT_DIR      = os.path.join(BASE_DIR, 'out')BRANCH = 'solid-architecture'   # nhánh có bước nhạcif os.path.exists(PROJECT_DIR):    shutil.rmtree(PROJECT_DIR)os.chdir(BASE_DIR)!git clone --depth 1 --branch {BRANCH} https://github.com/foresst123/sommerlier.gitassert os.path.exists(PIPELINE_DIR), "clone hong"print("commit:", os.popen(f"git -C {PROJECT_DIR} log -1 --oneline").read().strip())

## 3. Cài thư viện (~8–12 phút)Lấy thẳng `requirements.txt` của repo rồi **bỏ hai dòng** mà bước nhạc không đụng tới:- `nemo-toolkit[asr]` — chỉ `models/sortformer.py` cần, mà file đó đang bị comment trong `model_loader.py`- `git+.../DiariZen.git` — chỉ tiến trình con của diarization import, mà diarization đang tắtHai dòng đó chiếm phần lớn thời gian cài. Phần còn lại vẫn phải có: `main.py` import`whisperx`, `faster_whisper`, `pyannote.audio`, `pandas`, `pydub` ngay ở tầng module,nên thiếu một cái là chết trước khi chạm tới nhạc.

In [ ]:
!pip install -q uvreq_src = os.path.join(PIPELINE_DIR, 'requirements.txt')lines = [l for l in open(req_src).read().splitlines()         if not l.startswith('nemo-toolkit') and 'DiariZen.git' not in l]req_file = os.path.join(BASE_DIR, 'requirements_music.txt')open(req_file, 'w').write("\n".join(lines))print("\n".join(l for l in lines if l and not l.startswith('#')))

In [ ]:
!uv venv --allow-existing --python 3.11 {ENV_DIR}# Torch trước, CUDA 12.6 — để uv không tự kéo bản CPU khi giải phụ thuộc.!uv pip install --python {ENV_DIR} torch==2.7.1 torchaudio==2.7.1 torchvision==0.22.1 \    --extra-index-url https://download.pytorch.org/whl/cu126!uv pip install --python {ENV_DIR} -r {req_file} \    --extra-index-url https://download.pytorch.org/whl/cu126 --index-strategy unsafe-best-match

In [ ]:
python_bin = os.path.join(ENV_DIR, 'bin', 'python')check = """import torch, librosa, soundfile, pandas, pydub, onnxruntimeimport pyannote.audio, whisperx, faster_whisperfrom panns_inference import SoundEventDetectionfrom audio_separator.separator import Separatorprint("torch", torch.__version__, "| cuda", torch.cuda.is_available(), torch.cuda.device_count())print("moi thu can cho buoc nhac da san sang")"""# Ghi ra file roi chay: chuyen chuoi nhieu dong qua `!` thi dau nhay bi shell an mat.probe = os.path.join(BASE_DIR, 'probe_imports.py')open(probe, 'w').write(check)import subprocessr = subprocess.run([python_bin, probe], capture_output=True, text=True)print(r.stdout or r.stderr[-3000:])assert r.returncode == 0, "thieu thu vien — xem loi o tren"

## 4. Token HuggingFace (tuỳ chọn)Bước nhạc **không cần** token: PANNs tải checkpoint từ Zenodo, BS-RoFormer tải từ repo UVR công khai.Chạy ô này nếu bạn định chạy tiếp diarization sau đó.

In [ ]:
hf_token = ""try:    from kaggle_secrets import UserSecretsClient    s = UserSecretsClient()    for key in ("HUGGINGFACE_TOKEN", "HF_TOKEN"):        try:            hf_token = s.get_secret(key)            if hf_token: break        except Exception:            passexcept Exception:    passif hf_token:    os.environ["HUGGINGFACE_TOKEN"] = hf_token    print("Da lay token tu Kaggle Secrets.")else:    print("Khong co token — buoc nhac van chay binh thuong.")

## 5. Audio đầu vàoGắn dataset audio vào notebook (Add Input), ô này tự copy sang thư mục ghi được.Không có file nào thì nó dựng một bản ghi tổng hợp 60 giây: **nói → hát → nhạc nền dưới lời nói → nói**,đủ để thấy cả ba nhãn và cả hai hành vi (bóc nền, cắt đoạn hát).

In [ ]:
import glob, shutilimport numpy as npos.makedirs(AUDIO_DIR, exist_ok=True)found = []if os.path.exists('/kaggle/input'):    for ext in ('*.mp3','*.wav','*.flac','*.m4a','*.aac','*.ogg'):        found += glob.glob(f'/kaggle/input/**/{ext}', recursive=True)for f in found:    dst = os.path.join(AUDIO_DIR, os.path.basename(f))    if not os.path.exists(dst):        shutil.copy(f, dst)    print("  +", os.path.basename(f), f"({os.path.getsize(f)/1e6:.1f} MB)")audio_files = sorted(p for p in glob.glob(os.path.join(AUDIO_DIR, '*'))                     if p.lower().endswith(('.mp3','.wav','.flac','.m4a','.aac','.ogg')))if not audio_files:    print("Khong tim thay audio — dung ban ghi tong hop de thu luong.")    import soundfile as sf    sr = 24000    def voice(sec, f0=140.0):        t = np.arange(int(sec*sr))/sr        # nhieu thanh dieu + envelope am tiet: du de PANNs goi la Speech        f = f0 * (1 + 0.05*np.sin(2*np.pi*3.1*t))        sig = sum(np.sin(2*np.pi*k*np.cumsum(f)/sr)/k for k in (1,2,3,4))        env = 0.5 + 0.5*np.sin(2*np.pi*4.0*t)        return (sig*env/4).astype(np.float32)    def song(sec):        t = np.arange(int(sec*sr))/sr        notes = [261.6, 329.6, 392.0, 329.6]        f = np.concatenate([np.full(len(t)//4, n) for n in notes])[:len(t)]        return (0.3*np.sin(2*np.pi*np.cumsum(f)/sr)                + 0.2*np.sin(2*np.pi*np.cumsum(f*2)/sr)).astype(np.float32)    def bed(sec):        t = np.arange(int(sec*sr))/sr        return sum(0.08*np.sin(2*np.pi*f*t + p) for f, p in                   ((110,0),(165,1),(220,2),(330,3))).astype(np.float32)    parts = [voice(15), song(12), voice(18) + bed(18), voice(15)]    mix = np.concatenate(parts)    mix = (mix/np.max(np.abs(mix))*0.7).astype(np.float32)    path = os.path.join(AUDIO_DIR, 'synthetic_probe.wav')    sf.write(path, mix, sr)    audio_files = [path]print(f"\n{len(audio_files)} file san sang:")for p in audio_files:    print("  -", p)

## 6. Bật/tắt từng bước trong `config.json`Đây là cách dừng sau bước nhạc: **tắt `diarization`**. Diarization, ASR và export làba bước chịu lực — tắt một cái thì phía sau không còn đầu vào, nên pipeline ghi`01_music/` ra đĩa rồi dừng, kèm `manifest.json` nói rõ nó dừng trước bước nào.`music_removal_fallback` đang tắt sẵn: đó là bước 5 cũ (bóc nhạc lại lần nữa trêntừng segment sau separation), giờ không cần vì đã bóc ở mức sóng ngay đầu luồng.

In [ ]:
import json, pathlibcfg_path = os.path.join(PIPELINE_DIR, 'config.json')cfg = json.loads(pathlib.Path(cfg_path).read_text())profile = cfg['environments']['kaggle']profile['steps'].update({    "music_analysis":         True,    # PANNs SED gan nhan tung khung    "music_removal":          True,    # BS-RoFormer boc nen khoi doan `music`    "cut_singing":            True,    # cat `singing` + `song`, noi mep overlap-add    "music_removal_fallback": False,   # buoc 5 cu — tat    "diarization":            False,   # <-- DUNG O DAY})profile['models']['demucs']['model'] = 'bs_roformer'pathlib.Path(cfg_path).write_text(json.dumps(cfg, indent=2, ensure_ascii=False))print(json.dumps(profile['steps'], indent=2))print("music separator:", profile['models']['demucs'])

### Ngưỡng của PANNs (chỉnh nếu nghe thấy cắt sai)Chưa hiệu chỉnh trên dữ liệu thật — mới chỉ suy ra từ lý lẽ về chi phí sai lệch.Chạy lần đầu cứ để nguyên, nghe rồi mới chỉnh:| biến | mặc định | tăng lên khi | giảm xuống khi ||---|---|---|---|| `MUSIC_MAP_THRESHOLD` | 0.35 | bóc nhạc ở chỗ không có nhạc | sót nền nhạc || `MUSIC_MAP_SINGING` | 0.35 | cắt nhầm lời nói | sót đoạn hát || `MUSIC_MAP_SINGING_MARGIN` | 0.15 | nói-trên-nền bị gọi là hát | hát bị bỏ qua || `MUSIC_MAP_MIN_SPAN` | 0.30 s | span vụn | sót đoạn ngắn || `MUSIC_MAP_MERGE_GAP` | 0.50 s | một đoạn bị vỡ làm nhiều | hai đoạn bị dính |Có một chốt chặn: nếu bộ gán nhãn đòi cắt hơn **60%** bản ghi(`MUSIC_CUT_SHARE_LIMIT`), pipeline giữ nguyên audio và ghi cảnh báo — thà không cắtcòn hơn xoá mất bản ghi vì ngưỡng đặt sai.

In [ ]:
tuning = {    # "MUSIC_MAP_THRESHOLD":      "0.35",    # "MUSIC_MAP_SINGING":        "0.35",    # "MUSIC_MAP_SINGING_MARGIN": "0.15",    # "MUSIC_MAP_MIN_SPAN":       "0.30",    # "MUSIC_MAP_MERGE_GAP":      "0.50",}os.environ.update(tuning)print(tuning or "dung nguyen mac dinh")

## 7. ChạyChạy lại từ đầu thì bật `FRESH_RUN` — nếu không, checkpoint sẽ nạp lại bản đồ nhạc cũvà mọi ngưỡng vừa chỉnh ở trên đều không có tác dụng.

In [ ]:
FRESH_RUN = Trueif FRESH_RUN:    for d in (OUT_DIR, os.path.join(PIPELINE_DIR, 'cache')):        shutil.rmtree(d, ignore_errors=True)    # so ghi tien do: mot lan chay dung som van bi danh dau la xong,    # va lan chay day du sau do se bo qua file.    ledger = os.path.join(AUDIO_DIR, '_sommelier_progress.json')    if os.path.exists(ledger):        os.remove(ledger)    print("da xoa cache, output va so ghi tien do")

In [ ]:
import syssite = os.path.join(ENV_DIR, 'lib', 'python3.11', 'site-packages')os.environ["LD_LIBRARY_PATH"] = (os.environ.get("LD_LIBRARY_PATH", "")                                 + f":{site}/nvidia/cudnn/lib:{site}/torch/lib")os.chdir(PIPELINE_DIR)!CUDA_VISIBLE_DEVICES=0 {python_bin} main.py \    --audio_dir {AUDIO_DIR} \    --save_path {OUT_DIR} \    --env kaggle \    --lang vi \    --no_review_page

## 8. PANNs thấy gì`music_map.json` là toàn bộ phán quyết của bước này: mỗi span có `start`, `end`, `kind`tính theo **thời gian của bản gốc**, cộng với `timeline` cho biết phần nào còn lại sau khi cắt.

In [ ]:
import json, globruns = sorted(glob.glob(os.path.join(OUT_DIR, '*', '01_music', 'music_map.json')))assert runs, f"khong thay 01_music/music_map.json trong {OUT_DIR}"maps = {}for path in runs:    name = os.path.basename(os.path.dirname(os.path.dirname(path)))    payload = json.load(open(path))    maps[name] = payload    spans = payload.get('spans', [])    print(f"=== {name}")    print(f"    {len(spans)} span | nhac duoi loi noi {payload.get('music_seconds',0):.1f}s"          f" | hat {payload.get('singing_seconds',0):.1f}s"          f" | nhac khong loi {payload.get('song_seconds',0):.1f}s")    print(f"    cat di {payload.get('removed_seconds',0):.1f}s,"          f" con {payload.get('kept_stretches',0)} doan lien tuc")    for s in spans[:25]:        bar = {'music':'~~~', 'singing':'###', 'song':'***'}.get(s['kind'], '???')        print(f"      {bar} {s['kind']:8} {s['start']:8.2f} -> {s['end']:8.2f}"              f"  ({s['end']-s['start']:5.2f}s)")    if len(spans) > 25:        print(f"      ... con {len(spans)-25} span nua")    print()

## 9. Đóng gói audio để nghe trước/sau`clips/` là phần đáng nghe nhất: mỗi sự kiện được cắt hai đoạn cùng độ dài —một từ bản gốc, một từ bản đã xử lý, đặt cạnh nhau.- span **`music`**: nghe xem nền nhạc đã đi chưa, và tiếng nói còn nguyên không- span **`singing` / `song`**: đoạn đó đã biến mất; clip `after` là **mối nối**,  nghe xem có nghe thấy vết cắt không

In [ ]:
import numpy as np, soundfile as sf, librosaCLIP_PAD      = 4.0     # giay lay them moi benMAX_CLIPS     = 12      # moi loai spanINCLUDE_FULL  = True    # kem ca ban day du (FLAC)PACK = os.path.join(BASE_DIR, 'music_stage_pack')shutil.rmtree(PACK, ignore_errors=True)def to_cut(kept, t):    """Vi tri cua mot thoi diem goc trong ban da cat; None neu bi cat mat."""    if not kept:        return t    for a, b, c in kept:        if a <= t < b:            return c + (t - a)    return Nonedef grab(y, sr, centre, pad):    i = max(0, int((centre - pad) * sr))    j = min(len(y), int((centre + pad) * sr))    return y[i:j]for name, payload in maps.items():    src = next((p for p in audio_files                if os.path.splitext(os.path.basename(p))[0] == name), None)    after_path = os.path.join(OUT_DIR, name, '01_music', 'after_music.wav')    if src is None or not os.path.exists(after_path):        print(f"bo qua {name}: thieu nguon hoac after_music.wav")        continue    room = os.path.join(PACK, name)    os.makedirs(os.path.join(room, 'clips'), exist_ok=True)    shutil.copy(src, os.path.join(room, 'before_' + os.path.basename(src)))    json.dump(payload, open(os.path.join(room, 'music_map.json'), 'w'),              indent=2, ensure_ascii=False)    before, sr_b = librosa.load(src, sr=None, mono=True)    after,  sr_a = sf.read(after_path, dtype='float32')    if after.ndim > 1:        after = after.mean(axis=1)    if INCLUDE_FULL:        sf.write(os.path.join(room, 'after_music.flac'), after, sr_a,                 format='FLAC', subtype='PCM_16')    kept = (payload.get('timeline') or {}).get('kept', [])    counts = {}    for k, span in enumerate(payload.get('spans', [])):        kind = span['kind']        counts[kind] = counts.get(kind, 0) + 1        if counts[kind] > MAX_CLIPS:            continue        centre = 0.5 * (span['start'] + span['end'])        tag = f"{k:03d}_{kind}_{span['start']:.1f}s"        sf.write(os.path.join(room, 'clips', tag + '_before.wav'),                 grab(before, sr_b, centre, CLIP_PAD), sr_b)        # `music` van con trong ban da cat, nghe o dung cho no.        # `singing`/`song` da bi cat — nghe moi noi, tuc la diem bat dau cua no.        if kind == 'music':            at = to_cut(kept, centre)        else:            at = to_cut(kept, max(0.0, span['start'] - 0.05))        if at is None:            at = to_cut(kept, max(0.0, span['start'] - 0.5))        if at is not None:            sf.write(os.path.join(room, 'clips', tag + '_after.wav'),                     grab(after, sr_a, at, CLIP_PAD), sr_a)    made = len(os.listdir(os.path.join(room, 'clips')))    print(f"{name}: {made} clip"          f" | goc {len(before)/sr_b/60:.1f} phut -> sau {len(after)/sr_a/60:.1f} phut")zip_base = os.path.join(BASE_DIR, 'music_stage_audio')shutil.make_archive(zip_base, 'zip', PACK)print(f"\n{zip_base}.zip — {os.path.getsize(zip_base + '.zip')/1e6:.1f} MB")

### Tải vềKaggle: panel **Output** bên phải → `music_stage_audio.zip`. Ô dưới nghe thử ngay trong notebook.

In [ ]:
from IPython.display import Audio, displayimport globname = list(maps)[0]clips = sorted(glob.glob(os.path.join(PACK, name, 'clips', '*_before.wav')))[:4]for b in clips:    a = b.replace('_before.wav', '_after.wav')    print(os.path.basename(b).replace('_before.wav', ''))    print("  truoc:"); display(Audio(b))    if os.path.exists(a):        print("  sau:"); display(Audio(a))    else:        print("  sau: (doan nay da bi cat, khong co moi noi de nghe)")

## Nếu có gì đó không đúng| Triệu chứng | Nguyên nhân ||---|---|| `music_map.json` có 0 span | PANNs không thấy nhạc. Kiểm tra log có dòng `Music map:`; nếu không có, `steps.music_analysis` đang false || Cắt gần hết bản ghi | Chốt chặn 60% đã chặn lại và ghi cảnh báo trong log — nâng `MUSIC_MAP_SINGING` lên || Nghe rõ vết cắt ở mối nối | Fade 30 ms (`FADE_SECONDS` trong `utils/excise.py`). Dài hơn thì mượt hơn nhưng nhoè qua âm tiết bên cạnh — trung vị một chuỗi tiếng nói chỉ 0.40 s || Tiếng nói bị mỏng đi ở đoạn `music` | BS-RoFormer bóc quá tay. So `clips/*_music_*` trước/sau; nếu tệ, đổi `models.demucs.model` sang `demucs` để đối chứng || Chạy lại mà kết quả không đổi | Checkpoint. Bật `FRESH_RUN = True` || `CUDA out of memory` ở PANNs | Giảm `PANNS_SED_CHUNK` (mặc định 60 giây mỗi lần forward) |Chạy tiếp phần sau: bật lại `steps.diarization`, `separation`, `asr`, `export` trong ô 6.Lúc đó cần token HF **và** hai dòng đã bỏ ở ô 3 (`nemo-toolkit`, `DiariZen`).